# Nino Fine-Tuning (Open Images + optional custom photos)

This notebook trains a fresh EfficientDet-Lite0 object detector for the Nino assistive app. The training set is pulled automatically from the Open Images v7 dataset (already labeled, no manual work). Optionally you can merge your own labeled PascalVOC photos.

## How to run (no restart needed)
1. **Runtime -> Change runtime type -> T4 GPU** (recommended, much faster).
2. Press **Run all** and let it run. Do not close or background this tab; disable your laptop sleep.
3. The notebook switches the shell to Python 3.11 (required by the trainer) but never restarts the Colab kernel.

## Timing
- Steps 1-3 (switch to Python 3.11, install packages): ~3-5 min
- Step 4 (class check): ~30 sec
- Step 5 (download ~7,500 images from Open Images): 15-60 min
- Step 7 (train + export model): 30-90 min
- Step 8: downloads `nino_model.zip` automatically

## Troubleshooting
- A warning like `WARN SomeClass : 0 samples` is fine - that class just gets skipped.
- If cell 0 fails at the apt/PowerShell bit, tell me the exact message and I will fix the bootstrap.
- If the tab disconnects (`await connected: disconnected`), click **Connect**, then **Run all** again. Already-downloaded data is cached, so it resumes quickly.

In [ ]:
# --- 0. Switch the shell's python3 to Python 3.11 (required by mediapipe-model-maker) ---
# IMPORTANT: no runtime restart is needed. The notebook kernel stays as-is;
# every heavy step below runs as a subprocess under /usr/bin/python3 (now 3.11).
!sudo apt-get update -y -qq
!if ! command -v python3.11 >/dev/null 2>&1; then sudo add-apt-repository -y ppa:deadsnakes/ppa && sudo apt-get update -y -qq; fi
!sudo apt-get install -y -qq python3.11 python3.11-distutils
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
!sudo update-alternatives --set python3 /usr/bin/python3.11
!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py && /usr/bin/python3 /tmp/get-pip.py -q
!/usr/bin/python3 --version

In [ ]:
!/usr/bin/python3 --version
!/usr/bin/python3 -m pip --version

In [ ]:
!/usr/bin/python3 -m pip install -q --upgrade pip
!/usr/bin/python3 -m pip install -q mediapipe-model-maker
!/usr/bin/python3 -m pip install -q fiftyone
!/usr/bin/python3 -m pip install -q opencv-python-headless pandas

## 1. Choose your classes

Edit the `CLASSES` list in the next cell to add or remove class names. Names must match the Open Images dataset exactly; the step after this cell checks them for you.

Common extras you can add: `Dog`, `Cat`, `Bird`, `Boat`, `Train`, `Aircraft`, `Sheep`, `Horse`, `Cattle`, `Flower`, `Fountain`, `Fireplace`, `Mug`, `Knife`, `Fork`, `Spoon`, `Bowl`, `Wine glass`, `Pizza`, `Banana`, `Apple`, `Sandwich`, `Broccoli`, `Carrot`, `Hot dog`, `Remote control`, `Microwave oven`, `Houseplant`.

In [ ]:
import json

CLASSES = [
    "Person", "Bicycle", "Car", "Motorcycle", "Bus", "Truck", "Traffic sign", "Traffic light",
    "Fire hydrant", "Boat", "Train", "Aircraft", "Dog", "Cat", "Bird", "Horse", "Sheep", "Cattle",
    "Laptop", "Desktop computer", "Mobile phone", "Tablet computer", "Television", "Chair", "Table",
    "Sofa bed", "Bed", "Wardrobe", "Door", "Window", "Fence", "Stairs", "Escalator",
    "Building", "House", "Sidewalk", "Pavement", "Road", "Billboard", "Flag",
    "Bench", "Pergola", "Tree", "Houseplant", "Flower", "Fountain", "Fireplace", "Clock",
    "Umbrella", "Backpack", "Luggage and bags", "Wheelchair", "Stretcher", "Microwave oven",
    "Refrigerator", "Oven", "Dishwasher", "Sink", "Toilet", "Couch", "Picture frame",
]

config = {
    "classes": CLASSES,
    "max_samples_per_class": 120,   # images per class pulled from Open Images (auto-labeled)
    "val_split": 0.10,              # fraction held out for validation
    "epochs": 30,
    "batch_size": 8,
    "image_size": 320,
    "seed": 42,
    "skip_custom_data": True,       # set False if you also want to merge your own PascalVOC photos
}

with open("/content/nino_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(len(CLASSES), "classes configured")
print(CLASSES)

## 2. Validate class names

Run the two cells below. The first writes the validation script, the second executes it under Python 3.11.

In [ ]:
%%writefile /content/step1_validate.py
import json
import subprocess
import pandas as pd

cfg = json.load(open("/content/nino_config.json"))
classes = cfg["classes"]

subprocess.run(
    ["curl", "-sL",
     "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions.csv",
     "-o", "/content/oid_classes.csv"],
    check=True,
)
df = pd.read_csv("/content/oid_classes.csv", names=["id", "name"])
valid = set(df["name"])

missing = [c for c in classes if c not in valid]
print(len(classes), "classes requested")
if missing:
    print("INVALID class names - fix CLASSES in the config cell above:")
    for m in missing:
        print("  -", m)
    raise SystemExit(1)
print("All class names are valid.")

In [ ]:
!/usr/bin/python3 /content/step1_validate.py

## 3. Download the Open Images data

This downloads up to `max_samples_per_class` labeled images per class (already annotated, no labeling work) and saves a deduplicated snapshot. It is the longest download step - keep the tab open.

In [ ]:
%%writefile /content/step2_download.py
import json
import fiftyone as fo
import fiftyone.zoo as foz

cfg = json.load(open("/content/nino_config.json"))
classes = cfg["classes"]
max_samples = cfg["max_samples_per_class"]
seed = cfg["seed"]

sets = []
for c in classes:
    name = "nino_" + c.lower().replace(" ", "_")
    try:
        ds = foz.load_zoo_dataset(
            "open-images-v7",
            split="train",
            label_types=["detections"],
            classes=[c],
            max_samples=max_samples,
            shuffle=True,
            seed=seed,
            dataset_name=name,
        )
        if len(ds) == 0:
            print("WARN", c, ": 0 samples - skipping")
        else:
            sets.append(ds)
            print(c, ":", len(ds), "samples")
    except Exception as e:
        print("SKIP", c, ":", e)

if not sets:
    raise RuntimeError("No class data could be downloaded. Check class names and network, then rerun.")

dataset = sets[0].concat(sets[1:])

seen, keep = set(), []
for sample in dataset:
    if sample.filepath not in seen:
        seen.add(sample.filepath)
        keep.append(sample.id)
dataset = dataset.select(keep)

print("Total unique images:", len(dataset))
print("Downloading image files and saving a dataset snapshot...")
dataset.export("/content/nino_ds", dataset_type=fo.types.FiftyOneDataset)
print("Saved to /content/nino_ds")

In [ ]:
!/usr/bin/python3 /content/step2_download.py

## 4. (Optional) Merge your own photos

If `skip_custom_data` in the config cell is `False`, the next cell lets you upload a PascalVOC zip (folders `Annotations` + `JPEGImages`) of photos you labeled yourself. With `skip_custom_data=True` (default) this cell is skipped automatically.

In [ ]:
import json
import os
import zipfile
from google.colab import files

cfg = json.load(open("/content/nino_config.json"))
custom_dir = "/content/custom_voc"

if cfg["skip_custom_data"]:
    print("skip_custom_data=True - no custom photos needed, using Open Images data only.")
else:
    os.makedirs(custom_dir, exist_ok=True)
    print("Upload your PascalVOC zip (folders Annotations + JPEGImages):")
    up = files.upload()
    if up:
        zpath = list(up.keys())[0]
        with zipfile.ZipFile(zpath) as z:
            z.extractall(custom_dir)
        print("Extracted to", custom_dir)
        print(os.listdir(custom_dir))

Split into train/val and export PascalVOC for the trainer:

In [ ]:
%%writefile /content/step3_merge_export.py
import json
import os
import random
import fiftyone as fo

cfg = json.load(open("/content/nino_config.json"))
classes = cfg["classes"]
val_split = cfg["val_split"]
seed = cfg["seed"]

dataset = fo.Dataset.from_dir("/content/nino_ds", dataset_type=fo.types.FiftyOneDataset)
print("Loaded", len(dataset), "Open Images samples")

annot_dir = "/content/custom_voc/Annotations"
if os.path.isdir(annot_dir):
    custom_ds = fo.Dataset.from_dir("/content/custom_voc", dataset_type=fo.types.VOCDetectionDataset)
    dataset = dataset.concat(custom_ds)
    seen, keep = set(), []
    for sample in dataset:
        if sample.filepath not in seen:
            seen.add(sample.filepath)
            keep.append(sample.id)
    dataset = dataset.select(keep)
    print("Merged custom photos, total", len(dataset))
else:
    print("No custom data found - continuing with Open Images only.")

ids = list(dataset.values("id"))
random.seed(seed)
random.shuffle(ids)
n_val = max(1, int(len(ids) * val_split))
val_ids = set(ids[:n_val])
train_ids = set(ids[n_val:])

train_view = dataset.select(train_ids)
val_view = dataset.select(val_ids)

train_view.export(
    "/content/voc_train", dataset_type=fo.types.VOCDetectionDataset,
    label_field="detections", classes=classes,
)
val_view.export(
    "/content/voc_val", dataset_type=fo.types.VOCDetectionDataset,
    label_field="detections", classes=classes,
)
print("Exported VOC: /content/voc_train | /content/voc_val")

In [ ]:
!/usr/bin/python3 /content/step3_merge_export.py

## 5. Train EfficientDet-Lite0

This is the long step (~30-90 min on a T4 GPU). It trains on your class set and exports:
- `/content/detect.tflite` - the quantized model the Nino app will run
- `/content/labels.txt` - the class list (line 0 = class 0, so the app's `labelOffset` becomes 0)

In [ ]:
%%writefile /content/step4_train.py
import json
import tempfile
from mediapipe_model_maker import object_detector

cfg = json.load(open("/content/nino_config.json"))
epochs = cfg["epochs"]
batch_size = cfg["batch_size"]
image_size = cfg["image_size"]

train_data = object_detector.Dataset.from_pascal_voc_folder(
    annotation_dir="/content/voc_train/Annotations",
    image_dir="/content/voc_train/JPEGImages",
)
val_data = object_detector.Dataset.from_pascal_voc_folder(
    annotation_dir="/content/voc_val/Annotations",
    image_dir="/content/voc_val/JPEGImages",
)
print("train samples:", len(train_data), "| val samples:", len(val_data))

spec = object_detector.EfficientDetLite0Spec(model_dir=tempfile.mkdtemp(), image_size=image_size)
try:
    spec.hparams.epochs = epochs
    spec.hparams.batch_size = batch_size
except AttributeError:
    spec.epochs = epochs
    spec.batch_size = batch_size

print("Training EfficientDet-Lite0 ... (this is the long step)")
model = object_detector.EfficientDetLite0.create(train_data, model_spec=spec, validation_data=val_data)
model.evaluate(val_data)
model.export_to_tflite("/content/detect.tflite")
model.export_labels("/content/labels.txt")

labels = open("/content/labels.txt").read().splitlines()
print("Trained labels (%d):" % len(labels))
print(labels)
print("Model saved to /content/detect.tflite")

In [ ]:
!/usr/bin/python3 /content/step4_train.py

## 6. Download the trained model

In [ ]:
!zip -r -j /content/nino_model.zip /content/detect.tflite /content/labels.txt
from google.colab import files
files.download("/content/nino_model.zip")
print("Downloaded. Send me nino_model.zip and I will install it into the Nino app.")

## Done!

`nino_model.zip` contains `detect.tflite` and `labels.txt`. Send it to me and I will install it into the app (labels use offset 0), rebuild, and deploy it to your phone.